# 06 - Comparaison avec une extraction automatique de variables (tsfresh)

**Question** : des centaines de descripteurs génériques calculés automatiquement (tsfresh) font-ils mieux
que nos variables construites « à la main » avec une justification physique ?

**Comparaison équitable** : mêmes fenêtres (cœur de 1 s), même tâche, mêmes chiens, même protocole
(CV groupée, choix par CV, test à la fin).

**Contraintes de calcul** : tsfresh est lent. On l'applique aux **4 normes** et à un **sous-échantillon du
train** (au plus `MAX_PAR_CLASSE` fenêtres par classe) ; le test est complet. Pour rester équitable,
l'approche 1 est **réentraînée sur le même sous-échantillon**.

| Expérience | Variables | Entraînement |
|---|---|---|
| `approche1_sous_echantillon` | jeu de variables du modèle général retenu | sous-échantillon |
| `tsfresh_1s` | tsfresh sur le cœur de 1 s | sous-échantillon |
| `approche1+tsfresh` | les deux | sous-échantillon |

In [3]:
import sys
from pathlib import Path

# Racine du projet ajoutée au chemin d'import (le notebook est dans notebooks/)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config as cfg
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

import json
from src import data, labels, modeling, evaluation, viz
from src import tsfresh_features as tsf

In [4]:
windows = data.load_table(cfg.WINDOWS_FILE)
dogs = data.load_dogs()
train_ids, test_ids = data.load_split()
pointeur = json.loads((cfg.TABLES_DIR / "meilleur_modele_general.json").read_text())
general = modeling.load_bundle(**pointeur)
classes = list(general["label_encoder"].classes_)

# Même tâche et même filtrage qu'au notebook 04
task = labels.select_task_windows(windows, "general", min_purity=cfg.MIN_PURITY)
task = task.merge(dogs[["DogID"] + cfg.STATIC_FEATURES], on="DogID", how="left")
task = task[task["target"].isin(classes)]
train, test = task[task["DogID"].isin(train_ids)], task[task["DogID"].isin(test_ids)]
print(f"Modèle général : {general['model_name']} / {general['experiment']} | {len(classes)} classes")
print(f"Train : {len(train):,} fenêtres | Test : {len(test):,} fenêtres")

Modèle général : XGBoost / multi_echelle | 11 classes
Train : 93,124 fenêtres | Test : 27,654 fenêtres


In [5]:
MAX_PAR_CLASSE = 1500           # fenêtres de train par classe pour tsfresh
REGLAGES = "efficient"          # "efficient" (~800 variables par signal) ou "minimal" (test rapide)
N_JOBS_TSFRESH = 4
MODELES = ["Dummy_majoritaire", "RandomForest", "XGBoost"]

train_sub = tsf.subsample_per_class(train, MAX_PAR_CLASSE)
print(f"Sous-échantillon de train : {len(train_sub):,} fenêtres")
display(train_sub["target"].value_counts().to_frame("n_fenetres"))

Sous-échantillon de train : 15,095 fenêtres


,n_fenetres
target,
Walking,1500
Panting,1500
Trotting,1500
Lying chest,1500
Playing,1500
Standing,1500
Sniffing,1500
Eating,1500
Sitting,1500


## 1. Extraction tsfresh (mise en cache sur disque)

In [ ]:
cache_tr = cfg.PROCESSED_DIR / f"tsfresh_{REGLAGES}_train_{MAX_PAR_CLASSE}.parquet"
cache_te = cfg.PROCESSED_DIR / f"tsfresh_{REGLAGES}_test.parquet"
if cache_tr.exists() and cache_te.exists():
    X_tr_raw = data.load_table(cache_tr).set_index("window_id")
    X_te_raw = data.load_table(cache_te).set_index("window_id")
else:
    moves = data.load_moves(columns=["DogID", "TestNum", "t_sec"] + cfg.SENSOR_COLUMNS)
    X_tr_raw = tsf.extract_tsfresh(tsf.build_long_format(moves, train_sub), REGLAGES, N_JOBS_TSFRESH)
    X_te_raw = tsf.extract_tsfresh(tsf.build_long_format(moves, test), REGLAGES, N_JOBS_TSFRESH)
    data.save_table(X_tr_raw.reset_index(), cache_tr)
    data.save_table(X_te_raw.reset_index(), cache_te)
    del moves

X_tr, X_te = tsf.impute_from_train(X_tr_raw, X_te_raw)   # imputation apprise sur le train seulement
TSF_COLS = list(X_tr.columns)
print(f"{len(TSF_COLS)} variables tsfresh (après retrait des constantes)")
train_sub = train_sub.merge(X_tr, left_on="window_id", right_index=True, how="inner")
test_t = test.merge(X_te, left_on="window_id", right_index=True, how="inner")

## 2. Entraînement et comparaison (validation croisée)

In [ ]:
experiences = {
    "approche1_sous_echantillon": general["features"],
    "tsfresh_1s": TSF_COLS,
    "approche1+tsfresh": general["features"] + TSF_COLS,
}
resumes, meilleurs = [], {}
for exp, cols in experiences.items():
    resume, b = modeling.run_experiment(train_sub, cols, experiment=exp, task="general", models=MODELES)
    resumes.append(resume)
    meilleurs[exp] = modeling.best_bundle(b)
resultats_cv = pd.concat(resumes, ignore_index=True)
resultats_cv.to_csv(cfg.TABLES_DIR / "06_resultats_cv_tsfresh.csv", index=False)
display(resultats_cv.drop(columns=["meilleurs_params"]).round(3))

## 3. Évaluation sur le test (mêmes fenêtres pour toutes les expériences)

In [ ]:
lignes = []
for exp, b in meilleurs.items():
    pred, _ = modeling.predict(b, test_t)
    lignes.append({"experience": exp, "modele": b["model_name"], "f1_macro_cv": b["cv_mean"],
                   "f1_macro_cv_std": b["cv_std"],
                   **evaluation.compute_metrics(test_t["target"], pred, labels=classes)})
resultats_test = pd.DataFrame(lignes)
resultats_test.to_csv(cfg.TABLES_DIR / "06_resultats_test_tsfresh.csv", index=False)
display(resultats_test.round(3))

b = meilleurs["tsfresh_1s"]
retenues = modeling.selected_features(b["model"], b["features"])
print(f"tsfresh : {len(retenues)} variables retenues par la sélection. Exemples :", retenues[:15])

**Éléments de discussion** :
- performance : l'extraction générique bat-elle (ou complète-t-elle) les variables expertes ?
- coût : temps d'extraction (minutes à heures) contre quelques secondes pour l'approche 1 ;
- interprétabilité : nos variables ont un sens physique (orientation, fréquence de pas...) ; beaucoup de
  variables tsfresh sont difficiles à interpréter ;
- tsfresh ne voit ici que le cœur de 1 s : l'approche 1 multi-échelle dispose de plus de contexte. C'est un
  choix de coût à mentionner.